# Calibrated Data Check Observer's Notebook

This notebook performs various checks on a MeerKLASS UHF data block after flagging and calibration with synchrotron model. 

It assumes some understanding of MeerKLASS observation strategies, data products and analysis choices although brief explanations are provided. Further questions should be posted to museek-pipeline Slack channel or consult the quick guide pinned to the channel.

## Report Form
Submit this report form after reviewing this notebook on each data block

https://docs.google.com/forms/d/e/1FAIpQLSc3rmH_jAhCU2TEk9_eYmmC44LiMA27DcG_X7NtrDcf0CB7EA/viewform

In [ ]:
# == Notebook Setup ==
# The new 3 cells handle libraries importing and setup global parameters and functions
# for calculations in this notebook. They are intentally placed here, outside sections,
# and will be hidden once the notebook is converted to HTML with --no-input option.

# -- Import libraries --
# Standard libraries
import logging
import random
import sys
import warnings
from pathlib import Path

# Astro libraries
import healpy as hp
import matplotlib as mpl
import numpy as np
import numpy.typing as npt
import pysm3
import xarray as xr
from astropy import __version__ as astropy_version
from astropy import units as u
from astropy.coordinates import SkyCoord
from matplotlib import pyplot as plt
from matplotlib import ticker
from matplotlib.offsetbox import AnchoredText
from scipy.stats import ConstantInputWarning, spearmanr
from sklearn.linear_model import HuberRegressor

# Museek libraries
from museek import __version__ as museek_version
from museek.util import notebook_helper

In [ ]:
# -- Plotting, logging, notebook display configuration --
# Update matplotlib default parameters
mpl_params = {
    # Set font family to Sans-serif for better readability on computer
    "font.family": "sans-serif",
    # Optional: Specify preferred sans-serif fonts (fallback order)
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "font.size": 12,
    # Make sure the figure background is white
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    # Always turn on minor ticks
    "ytick.minor.visible": True,
    "xtick.minor.visible": True,
}
mpl.rcParams.update(mpl_params)

# Configure xarray to expand data variables by default when displaying datasets to
# make sure all variables are displayed when converting the notebook to HTML
xr.set_options(display_expand_data_vars=True, display_max_rows=200)

# Setup logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    logger.addHandler(handler)

# Suppress healpy and pysm3 verbosity to WARNING only
logging.getLogger("pysm3").setLevel(logging.WARNING)
logging.getLogger("healpy").setLevel(logging.WARNING)

In [ ]:
# -- Global variables and functions for plotting --
def add_anchored_text(
    ax,
    text,
    loc="upper left",
    size="medium",
    alpha=0.5,
    edgecolor="lightgrey",
    facecolor="white",
):
    """Add anchored text to a given axis."""
    anchored_text = AnchoredText(
        text,
        loc=loc,  # Exact same location strings as legend
        prop=dict(size=size),  # Font size of the text
        borderpad=0.3,  # Fixed physical gap from axis edge
        frameon=True,  # Set False if you want a invisible background box
    )
    anchored_text.patch.set_alpha(alpha)
    anchored_text.patch.set_facecolor(facecolor)
    anchored_text.patch.set_edgecolor(edgecolor)

    ax.add_artist(anchored_text)


def scatter_sky(
    ax: mpl.axes.Axes,
    ra_arr: np.ndarray,
    dec_arr: np.ndarray,
    values: np.ndarray,
    vmin: float | None = None,
    vmax: float | None = None,
    cbar: bool = False,
    cbar_label: str | None = None,
    point_sources: dict | None = None,
):
    """Plot sky values at (RA, Dec) coordinates with point sources overlaid.

    `ra_arr` is expected to already be unwrapped (e.g. via
    `notebook_helper.unwrap_ra_deg`) by the caller if it spans the 360/0 branch
    cut, since only the caller knows whether `ra_arr` is a continuous,
    time-ordered sequence for which unwrapping is valid. `point_sources`, being
    an unordered catalog subset, is instead shifted onto the same 360-degree
    branch as `ra_arr` here, since that works regardless of ordering.

    Returns the `PathCollection` scatter artist so callers can build their own
    (e.g. shared, multi-axes) colorbar instead of using `cbar=True`.
    """
    sc = ax.scatter(
        ra_arr,
        dec_arr,
        c=values,
        edgecolor="none",
        cmap="jet",
        vmin=vmin,
        vmax=vmax,
        rasterized=True,
    )
    if cbar:
        ax.get_figure().colorbar(sc, ax=ax, label=cbar_label)
    if point_sources is not None:
        ref = float(np.mean(ra_arr))
        ra_1jy = notebook_helper.wrap_to_nearest(point_sources["ra_1jy"], ref)
        ra_5jy = notebook_helper.wrap_to_nearest(point_sources["ra_5jy"], ref)
        ax.scatter(
            ra_1jy,
            point_sources["dec_1jy"],
            facecolors="none",
            edgecolors="black",
            marker="s",
            s=40,
            rasterized=True,
        )
        ax.scatter(
            ra_5jy,
            point_sources["dec_5jy"],
            color="black",
            marker="+",
            s=70,
            rasterized=True,
        )
    return sc


def get_synchrotron_map(
    freq: float,
    nside: int = 128,
    models=["s1"],
    smooth: bool = True,
    beamsize: float = 57.5,
    beam_ref_freq: float = 1500.0,
) -> npt.NDArray:
    "Generate a synchrotron map at a given frequency using PySM3, optionally smooth it"
    "with a Gaussian beam."
    sky = pysm3.Sky(nside=nside, preset_strings=models)
    map = sky.get_emission(freq * u.MHz).value
    if smooth:
        # smooth with Gaussian beam scaled to freq_plot
        fwhm = beamsize * u.arcmin * (beam_ref_freq * u.MHz / (freq * u.MHz))
        map_smooth = pysm3.apply_smoothing_and_coord_transform(map, fwhm=fwhm)[0]
        return map_smooth
    else:
        return map


def get_healpix_interp_vals(healpix_map, ra, dec):
    """Get interpolated values from a Healpix map at given RA and Dec coordinates."""
    c = SkyCoord(ra=ra * u.degree, dec=dec * u.degree, frame="icrs")
    theta = (np.pi / 2) - c.galactic.b.rad  # type: ignore[operator]
    phi = c.galactic.l.rad  # type: ignore[union-attr]
    vals = hp.pixelfunc.get_interp_val(healpix_map, theta, phi).value
    return vals


def load_point_sources_for_ds(
    ds: xr.Dataset, antenna: str = "m002", catalog_dir: str | Path | None = None
) -> dict:
    """Load 1 Jy and 5 Jy point sources within the RA/Dec bounding box actually
    covered by the (RA, Dec) range in the dataset.

    A "good" antenna should be specified to avoid outliers in the RA/Dec range
    due to bad pointing.

    """
    ra_vals = ds["ra"].sel(antennas=antenna).values
    dec_vals = ds["dec"].sel(antennas=antenna).values
    # RA wraps at 360/0, so a plain mean/min/max is wrong whenever the track crosses
    # that boundary; use a circular mean and wrap the track onto its nearest branch
    # before taking min/max to get a sane center/radius.
    ra_center = notebook_helper.circular_mean_deg(ra_vals)
    dec_center = float(dec_vals.mean())
    ra_wrapped = notebook_helper.wrap_to_nearest(ra_vals, ra_center)
    ra_radius = float(np.max(np.abs(ra_wrapped - ra_center)))
    dec_radius = max(abs(dec_center - dec_vals.min()), abs(dec_center - dec_vals.max()))
    return notebook_helper.load_point_sources(
        ra_center=ra_center,
        dec_center=dec_center,
        ra_radius=ra_radius,
        dec_radius=dec_radius,
        catalog_dir=catalog_dir,
    )


ANTENNAS_PER_FIG = 8

## Version Information

In [ ]:
# Print museek version
print("Software versions used in this notebook:")
print(f"Python: {sys.version}")
print(f"Numpy: {np.__version__}")
print(f"xarray: {xr.__version__}")
print(f"astropy: {astropy_version}")
print(f"healpy: {hp.__version__}")
print(f"pysm3: {pysm3.__version__}")
print(f"museek: {museek_version}")

## 1. Data Preparation

Read raw and calibrated visibilities and flags from `aoflagger_plugin_postcalibration.pickle` and build an [xarray](https://xarray.dev/) dataset to standardise the data format and help with memory usage in calculations in this notebook.\

Familiarise yourself with the different flags in the data variables and dimensionality of this dataset. In summary, `raw_vis` is the uncalibrated  auto-correlation visitibility in `(timestamps, frequencies, antennas, feeds)`, and thus `feeds=h` implies `HH` polariation and vice versa. `cal_vis` is in `(timestamps, frequencies, antennas, stokes=I)`, formed by adding the h and v autos. `raw_flags_*` and `cal_flags_*` are flags that MuSEEK plugins produce for `raw_vis` and `cal_vis`. See [Museek's README](https://github.com/meerklass/museek#flags) for summary of each flag.

In [ ]:
# -- Default parameters --
# This cell has "parameters" tag with default data and path values neccesary to run
# the notebook. Papermill will use these values when executing the notebook if no
# overriden parameters are given through the papermill command. If overrides are given,
# a new cell will be injected below this cell with "injected-parameters" tag, with the
# desired parameters, overwritting values in this cell. The default parameters here can
# be inspected by calling `papermill --help-notebook <path/to/this/notebook>`.
block_name: str = "1779036710"  # Block name (CBID)
# block_name = "1778715926"
patch: str = "box14"  # Patch name, e.g. "box14"
base_context_folder: str = "/idia/projects/meerklass/MEERKLASS-1/museek"  # Base context folder. Notebook will look for the data in base_context_folder/patch/block_name/context
cache_file: str | None = (
    None  # Optional path to a cache the dataset as *.nc file for experimenting.
)

In [ ]:
# Build context file path.
context_dir = Path(base_context_folder) / patch / block_name / "context"
pickle_file = context_dir / "aoflagger_plugin_postcalibration.pickle"

# Load context data into an xarray Dataset, trimming frequency dimension.
# This should only use as much memory as the file size + notebook overhead.
# Apart from minimising memory footprint, putting the context data into xarray
# standardises the data structure as the pickle file contains a mixture of several
# data formats and some variables are sometimes not saved in the most logical places.
#
# If manually executing this notebook for testing, it is recommded to define and pass
# a `cache_file` parameter to `load_context_to_ds`, which will write the constructed
# datasets to `.nc` file on disk for faster subsequent loading. xarray dataset is
# lazy-loaded -- only metadata will be read into memory at the load time, and data
# variables will only be loaded into memory when accessed. This can help reduce memory
# usage greatly when experimenting with calculations. `skip_cache=True` can be used to
# force reloading the context data from the pickle file and overwriting the cache file.
#
# For pipeline runs, it is recommend to not use the cache file to not build up cache
# that is not used and tracked.
#
# Either define `cache_file` in the parameters above, or uncomment and modify the
# following lines
# cache_file = (
#     Path(os.getenv("XDG_CACHE_HOME", "~/.cache/museek")).expanduser()
#     / f"ds_cache_{block_name}.nc"
# )
load_context_kwargs = {
    "pickle_file": pickle_file,
    "frequency_range": "auto",
    "extra_attrs": {"patch": patch, "ds_cache_file": str(cache_file)}
    if cache_file is not None
    else {"patch": patch},
}
if cache_file is not None:
    load_context_kwargs["cache_file"] = cache_file
    load_context_kwargs["skip_cache"] = False
ds = notebook_helper.load_context_to_ds(**load_context_kwargs)

In [ ]:
# Print the dataset summary, note the shared dimensions and coordinates among the
# data variables. raw_vis, cal_vis, each flag, and RA/Dec are recorded as a separate
# data variable in this dataset.
print(ds)

## 2. Antenna Usability, Flag Fraction and Correlation with Synchrotron Model

Calculate and report antenna usability after each pipeline step. This is done by accumulating flags after pipeline steps, collapsing them over (timestamps, frequencies, feeds) dimensions with OR to yield per-antenna flags to check which antennas are fully flagged after each pipeline step. 

The top row of the figure shows per-antenna flagged fraction, representing the fraction of the data that has been flaggeg, excluding antennas not in the dataset (grey label) and fully-flagged antennas after calibration (red label).

The bottom row plots the Spearma's rank correlation coefficient between the calibrated visibility and the synchrotron model, $\rho_{synch} \in [-1, 1]$. A correlation value of 1 indicates a perfectly monotinic relationship, i.e. that the calibrated visibiliy perfectly matches the synchrotron model.

Note the followings when reviewing this figure:
* Ensure that the fully-flagged antennas are indeed flagged; that is, they do not have correlation values and are flagged in the [calibrated vis vs. synchrotron model](#8.-Median-Subtracted-Calibrated-Visibility-and-Synchrotron-Model-Sky-Scatter) figures later in this notebook 
* Note any antenna with a correlation lower than 0.75. Antennas with correlations between 0.5 and 0.75 are above the default threshold used to produce `cal_flags_synch_correlation_flag` (=0.5) and are therefore not flagged, but they may still be problematic.

In [ ]:
def compute_usability(ds) -> list[str]:
    """
    Compute and print the number of usable antennas after each pipeline step, and
    return the listof usable antennas after the last calibration step.

    To save memory:
    - Keep a single (timestamps, frequencies, antennas) bool array in memory.
    - Flags with feeds dimension are immediately collapsed to antennas (OR over H/V)
      before accumulation, halving the array size vs. keeping the feeds dim.
    - Accumulate with numpy in-place OR (`|=`) to avoid any allocation per step.
    - Only one temporary array (the per-step antenna-collapsed flag) exists
      alongside the cumulative; it is freed at the start of the next iteration.

    """
    meerkat_all_ants = ds.attrs["all_meerkat_antennas"]
    # Empty boolean array for storing cumulative flags. This is the same shape as
    # cal_flags_combined (excluding its size-1 stokes dim), which is
    # (timestamps, frequencies, antennas)
    cumulative = np.zeros(ds["cal_flags_combined"].isel(stokes=0).shape, dtype=bool)

    # List for storing result: (plugin_name, flag_name, usable_count)
    flag_summary = []

    # Iterate over each raw flag in the raw_flag_name_list, collapsing H/V to antennas
    for flag_name in ds.attrs["raw_flag_name_list"]:
        flag_da = ds[f"raw_flags_{flag_name}"]

        # If a feeds dim is present, OR-collapse H/V to antennas
        if "feeds" in flag_da.dims:
            flag_vals = flag_da.any(dim="feeds").values
        else:
            flag_vals = flag_da.values

        # Cumulate flags in-place with OR
        cumulative |= flag_vals

        # Collasing (timestamps, frequencies,) to yield (antennas,) flags
        per_ant_flags = cumulative.all(axis=(0, 1))

        # Sum over (antennas,) of the reversed flags to get the number of usable antennas
        num_usable_ants = int(np.sum(np.logical_not(per_ant_flags)))

        # Storing result
        plugin = notebook_helper.FLAG_PLUGIN_MAP.get(flag_name, flag_name)
        flag_summary.append((plugin, flag_name, num_usable_ants))
    del cumulative

    # For cal flags, we only need the number after the final calibration
    per_ant_flags_cal = notebook_helper.reduce_flags(
        flag_da=ds["cal_flags_combined"], output_dims=("antennas",), operator="and"
    ).values
    num_usable_ants = int(np.sum(np.logical_not(per_ant_flags_cal)))
    flag_summary.append(
        ("aoflagger_postcalibration_plugin", "cal_flags_combined", num_usable_ants)
    )

    # Print the names of antennas excluded in this observation at scheduled time
    # (not in the antennas list)
    ant_not_in_observation = [a for a in meerkat_all_ants if a not in ds.antennas]
    print("Total numbers of MeerKAT antennas:", len(meerkat_all_ants))
    print(
        "Antennas excluded in this observation at scheduled time:",
        len(ant_not_in_observation),
        ant_not_in_observation,
    )

    # Print result. Some plugins add two flags (e.g. antenna_flagger_plugin).
    # Print one row per plugin (antenna_flagger_plugin adds two flags — suppress the first).
    print("Numbers of usable antennas after each pipeline step:")
    for i, (plugin, flag_name, count) in enumerate(flag_summary):
        is_last_for_plugin = (
            i == len(flag_summary) - 1 or flag_summary[i + 1][0] != plugin
        )
        if is_last_for_plugin:
            print(f"  {plugin}: {count}")

    # Print the names of antennas fully masked in the final calibrated_vis
    # (after last calibration step)
    ants_flagged_final = ds.antennas[per_ant_flags_cal].values.tolist()
    print(
        "Antennas flagged after the last calibration step:",
        len(ants_flagged_final),
        ants_flagged_final,
    )
    good_ants = [a for a in ds.antennas.values.tolist() if a not in ants_flagged_final]
    return good_ants

In [ ]:
def plot_flag_fraction_and_r_vis(ds) -> None:
    """
    Plot the per-antenna flag fraction and correlation coefficient with Synchrotron
    model after the final calibration step.
    """
    # Calculate / extract flag fraction and correlation coefficient with Synchrotron model
    # This has dims=("antennas",)
    flag_fraction = ds["cal_flags_combined"].sum(dim=["timestamps", "frequencies"]).sel(
        stokes="I", drop=True
    ) / (ds.sizes["timestamps"] * ds.sizes["frequencies"])
    r_vis = ds["r_vis_synch_ant"]

    # We want to make plots with all antennas on the x-axis. Use xarray `reindex` to expand
    # "antennas" dimension to include all antennas, filling missing values with NaN.
    all_ants = ds.attrs["all_meerkat_antennas"]
    flag_fraction = flag_fraction.reindex(antennas=all_ants, fill_value=np.nan)
    ff_not_one = flag_fraction.where(flag_fraction < 1.0)
    r_vis = r_vis.reindex(antennas=all_ants, fill_value=np.nan)

    # Plot the data. Use rasterized=True to reduce file size when saving.
    # Also use a custom numeric antenna numbers as x to help with plotting the STD
    # shade region beyond the range of antennas.
    x = np.arange(0, 66)
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(8, 4), sharex="all", layout="constrained"
    )

    # -- Flag Fraction --
    # First plot the flag fraction
    ax1.scatter(x=x[1:-1], y=ff_not_one.values, rasterized=True)
    # Plot mean +/- std (excluding fully flagged antennas)
    ax1.axhline(
        y=ff_not_one.mean(skipna=True), color="C0", linestyle="--", label="Mean"
    )
    ax1.fill_between(
        x=x,
        y1=ff_not_one.mean(skipna=True) - ff_not_one.std(skipna=True),
        y2=ff_not_one.mean(skipna=True) + ff_not_one.std(skipna=True),
        color="C0",
        alpha=0.15,
        label="±1σ",
    )
    ax1.grid(visible=True, which="major", axis="x", linestyle=":")

    # -- Correlation Coefficient --
    # Then plot the correlation coefficient with Synchrotron model
    ax2.scatter(x=x[1:-1], y=r_vis.values, rasterized=True)
    # Plot mean +/- std (excluding fully flagged antennas)
    ax2.fill_between(
        x=x,
        y1=r_vis.mean(skipna=True) - r_vis.std(skipna=True),
        y2=r_vis.mean(skipna=True) + r_vis.std(skipna=True),
        color="C0",
        alpha=0.15,
        label="±1σ",
    )
    ax2.axhline(y=r_vis.mean(skipna=True), color="C0", linestyle="--", label="Mean")

    # Deal with grid and ticks. We want no minor ticks on the x-axis.
    ax2.xaxis.set_minor_locator(ticker.NullLocator())
    ax2.grid(visible=True, which="major", axis="x", linestyle=":")
    ax2.set_xlim(0, 64)
    # Relabel the x-axis with antenna names
    ax2.set_xticks(
        x,
        [
            "",
        ]
        + all_ants
        + [""],  # Prepend and append with "" to align labels with x
        rotation=90,
        fontsize="small",
    )
    # Recolor antenna labels not in the observation in grey
    ant_not_in_observation = [a for a in all_ants if a not in ds.antennas]
    ant_fully_flagged = flag_fraction.antennas[flag_fraction == 1.0].values.tolist()
    for label in ax2.get_xticklabels():
        if label.get_text() in ant_not_in_observation:
            label.set_color("grey")
        elif label.get_text() in ant_fully_flagged:
            label.set_color("red")
    fig.text(
        0.01,
        0.01,
        "Antennas excluded from this observation at scheduled time",
        fontsize="x-small",
        color="grey",
        ha="left",
    )
    fig.text(
        0.98,
        0.01,
        "Antennas fully flagged after calibration",
        fontsize="x-small",
        color="red",
        ha="right",
    )
    # Label the rest of the figure
    ax1.set_ylabel("Flagged Fraction")
    ax2.set_ylabel(r"$\rho_{synch}$")
    ax2.set_xlabel("Antenna Names")
    fig.suptitle(f"Usability Summary - {ds.attrs['block_name']}")
    fig.align_ylabels([ax1, ax2])
    # Legend: get handles and labels from the first axis only, and place a single
    # legend for the whole figure
    handles, labels = ax1.get_legend_handles_labels()
    fig.legend(handles, labels, ncol=2, loc="upper right", frameon=False)
    # fig.get_layout_engine().set(rect=[0, 0, 1, 0.95])

In [ ]:
good_ants = compute_usability(ds)
bad_ants = [a for a in ds.antennas.values.tolist() if a not in good_ants]
# Add good and bad ants to the dataset attributes for later use in plotting and analysis
ds.attrs["good_ants"] = good_ants
ds.attrs["bad_ants"] = bad_ants

In [ ]:
plot_flag_fraction_and_r_vis(ds)

## 3. Non-Linearity Contamination

Evaluate the Spearman's rank correlation between synchrotron-subtracted raw visibility and the RFI power over the GSM band (925-960 MHz) ($\rho_{RFI} \in [-1, 1]$). The strong GSM RFI causes the receiver gains to become non-linear, which manifests as zebra (stripy) patterns in the raw visibility. A value closer to +1 indicates more positive monotonic relationship, i.e. that the data is more correlated with the GSM RFI and are thus more likely to be contaminated by non-linear effects.

The left column of the figure shows per-antenna correlation values with the mean value (dashed grey) and ±1σ and ±2σ ranges (dark and light shades). The right column shows the density disbribution of the values. Note that the y-axis is not fixed to [-1, 1], only spaning the range of the values.

Note the mean and STD of the correlation values, as well as any antennas that can be considered as "outliers". Keep these numbers in mind while inspecting the [raw visibility sky scatter](#7.-Raw-Visibility-Sky-Scatter-before-AOFlagger) plots later in the notebook. It is currently unclear and may not be possible to specify a threshold for $\rho$ would cause zebra, but we want to develop a sense for that. 

Note: the test is only perform at 730 MHz with ["s1" PySM model](https://pysm3.readthedocs.io/en/latest/models.html#synchrotron) in this notebook as a quick QA test. In reality, the test results will depends on frequencies ad Synchrotron models and will thus be done over a range of frequencies.

In [ ]:
def zebra_correlation_test(
    ds: xr.Dataset,
    test_freq: float = 730.0,
) -> xr.DataArray:
    """
    Calculate correlation between synchrotron-subtracted raw visibility residuals and
    GSM RFI to test for non-linearities.

    High correlation (~>0.75) indicates that the raw visibility is dominated by GSM RFI
    and non-linearity artefacts (the zebras).

    To obtain the synchrotron-subtracted residuals, HuberRegressor, which is robust
    againts outliers, is used to fit a linear model between a chosen synchrotron model
    ("s1" model in PySM3 in this case) and the raw visibility. The fit coefficients are
    then used to subtract a linear synchrotron model from the raw visibility. Thus,
    the test results depend on the accuracy of the synchrotron model.

    Return Spearman correlation array of shape (n_antennas, n_feeds, 2)
    for (spearman_r, p_value).
    """
    # -- Data Preparation --
    # RFI power: use GSM 900 downlink band (925-960 MHz) as a proxy
    gsm_freq_slice = slice(925, 960)
    gsm = notebook_helper.select_and_flag(
        ds,
        var="raw_vis",
        flags="raw_flags_noise_diode_on",
        sel=dict(frequencies=gsm_freq_slice),
    ).sum(dim="frequencies", skipna=True)

    # Raw visibility
    # Apply noise diode flags and down select to test frequency
    vis_test = notebook_helper.select_and_flag(
        ds,
        var="raw_vis",
        flags="raw_flags_noise_diode_on",
        nearest=dict(frequencies=test_freq),
    )

    # Standaline noise diode flags of shape (timestamps,)
    # Calculate this once and reuse for all antennas/feeds when performing fit rather
    # than running select_and_flag on the vis_plot on every loop.
    # Noise-diode state is constant across antennas/feeds, so any (antenna, feed) works.
    nd_times = (
        ds["raw_flags_noise_diode_on"]
        .sel(frequencies=test_freq, method="nearest")
        .isel(antennas=0, feeds=0)
        .values
    )

    # Synchrotron model
    sky_model = get_synchrotron_map(test_freq)

    # Interpolate the synchrotron model at every antenna's RA/Dec track in one shot,
    # ravel/reshape inputs/outputs to get_interp_vals to avoid looping
    ra_flat = ds["ra"].values.ravel()
    dec_flat = ds["dec"].values.ravel()
    c = SkyCoord(ra=ra_flat * u.degree, dec=dec_flat * u.degree, frame="icrs")
    theta = (np.pi / 2) - c.galactic.b.rad
    phi = c.galactic.l.rad
    # Get interp values from PySM3 skymodel and convert to Kelvin
    synch_flat = hp.pixelfunc.get_interp_val(sky_model, theta, phi).value / 1e6
    # Reshape and putback into a dataarray of shape (timestamps, antennas)
    synch_I = ds["ra"].copy(data=synch_flat.reshape(ds["ra"].shape))

    # -- Main Calculations -- done per antenna and per feed
    # Fit a linear model between the raw_vis and synchrotron model, subtract the
    # fitted synchrotron model from the raw_vis, and calculate the spearman correlation
    # between the synchrotron-subtracted residuals and GSM RFI. HuberRegressor is used
    # for the fit as it robust againts outliers
    huber = HuberRegressor(
        epsilon=1.35,
        max_iter=100,
        alpha=0.0001,
        warm_start=False,
        fit_intercept=True,
        tol=1.0,
    )
    antennas = ds.coords["antennas"].values  # Conveneient variable
    # Array for storing results shape (n_antennas, n_feeds, 2) for (spearman_r, p_value)
    spearman = np.zeros((len(antennas), 2, 2), dtype=float)
    for i_ant, ant in enumerate(antennas):
        x = synch_I.sel(antennas=ant).values
        for i_feed, feed in enumerate(["h", "v"]):
            y = vis_test.sel(antennas=ant, feeds=feed).values
            valid = ~nd_times & np.isfinite(y)
            huber.fit(x[valid, None], y[valid])
            ysub = y[valid] - x[valid] * huber.coef_[0]
            gsm_valid = gsm.sel(antennas=ant, feeds=feed).values[valid]
            # Pick up scipy spearmanr's warning as error, so we can catch where it happens
            with warnings.catch_warnings():
                # Treat the warning as a hard error inside this block
                warnings.simplefilter("error", ConstantInputWarning)
                try:
                    spearman[i_ant, i_feed, :] = spearmanr(gsm_valid, ysub)
                except ConstantInputWarning:
                    spearman[i_ant, i_feed, :] = (np.nan, np.nan)
                    print(
                        f"Warning: Constant input encountered for antenna {ant}, feed {feed}. Spearman correlation set to NaN."
                    )

    # Return as xarry DataArray with coords and dims for easier downstream analysis and plotting
    coords = {
        "antennas": antennas,
        "feeds": ["h", "v"],
        "stats": ["spearman_r", "p_value"],
    }
    dims = ["antennas", "feeds", "stats"]
    return xr.DataArray(spearman, coords=coords, dims=dims)


spearman_skysub = zebra_correlation_test(ds)

In [ ]:
def plot_zebra_test(ds: xr.Dataset, spearman: xr.DataArray) -> None:
    x = ds.antennas
    # Build (H, V, Mean) of spearman values for easy plotting
    yy = (
        spearman[:, 0, 0],  # H feed spearman_r
        spearman[:, 1, 0],  # V feed spearman_r
        np.nanmean(spearman[:, :, 0], axis=1),  # Mean of H and V feeds
    )

    panel_labels = ["h", "v", "h & v Mean"]
    gridspec_kw = {"width_ratios": [10, 1], "hspace": 0.0, "wspace": 0.0}
    fig, axes = plt.subplots(
        3,
        2,
        figsize=(8, 5.5),
        sharex="col",
        sharey="row",
        gridspec_kw=gridspec_kw,
        layout="constrained",
    )

    for i, (ax, y, label) in enumerate(zip(axes, yy, panel_labels)):
        # Left column - main scatter plot
        ax[0].scatter(x, y, c=f"C{i}", marker="x", rasterized=True)
        ax[0].axhline(np.nanmean(y), color="grey", linestyle="--", label="Mean")
        # Turn off minor ticks on the x-axis
        ax[0].xaxis.set_minor_locator(ticker.NullLocator())
        ax[0].grid(visible=True, which="major", axis="x", linestyle=":")
        ax[0].set_ylabel(f"$\\rho_{{RFI}}$ ({label})")
        # Print mean and STD for each panel
        mean_val = np.nanmean(y)
        std_val = np.nanstd(y)
        ax[0].fill_between(
            x=x,
            y1=mean_val - std_val,
            y2=mean_val + std_val,
            color=f"C{i}",
            alpha=0.15,
            label="±1σ",
        )
        ax[0].fill_between(
            x=x,
            y1=mean_val - 2 * std_val,
            y2=mean_val + 2 * std_val,
            color=f"C{i}",
            alpha=0.10,
            label="±2σ",
        )
        text = f"Mean: {mean_val:.3f}\nSTD: {std_val:.3f}"
        add_anchored_text(ax[0], text, loc="lower right", size="small")

        # Plot half-violin (KDE) on the right
        ax[1].violinplot(
            y,
            positions=[0],
            side="high",
            showmeans=False,
            showextrema=False,
            facecolor=f"C{i}",
        )
        ax[1].axhline(np.nanmean(y), color="grey", linestyle="--", label="Mean")
        # Hide the right, top and bottom spines (borders)
        ax[1].get_xaxis().set_visible(False)
        for spine in ["top", "bottom", "right"]:
            ax[1].spines[spine].set_visible(False)
    # Set x-axis ticks and labels for the bottom row of scatter plots
    axes[-1, 0].tick_params(axis="x", which="major", rotation=90, labelsize="x-small")
    axes[-1, 0].set_xlabel("Antennas")
    fig.align_ylabels(axes[:, 0])
    fig.suptitle(
        f"Correlation between Synchrotron-subtracted Raw Visibility and GSM - {block_name}"
    )
    # Shrink xlim a bit
    xmin, xmax = axes[-1, 0].get_xlim()
    axes[-1, 0].set_xlim(xmin + 1, xmax - 1)


plot_zebra_test(
    ds, spearman_skysub
)  # Top: per-antenna line plot  # Bottom: boxplot with overlaid scatter  # arcmin FWHM at beam_frequency  # MHz

## 4. Time- and Frequency- Medians of Raw Visibility before AOFlagger

Examine the time and frequency medians of the raw visibilities before RFI flagging with AOFlagger was conducted. 

The following flags are applied to the raw vis before calculating the median:
* `raw_flags_SARAO`: Any flags added by SARAO SPD pipeline, e.g. an antenna that stops working midway during the observation
* `raw_flags_noise_diode_on`: Timestamps, where noise diode is being fired
* `raw_flags_known_rfi`: GSM900, GSM1800, GPS up/down links
* `raw_flags_rawdata_low_value`: Anyv raw data with values less than threshold (default 0.5 in raw correlator unit)

In [ ]:
# Build a mask for flags that existed before AOFlagger was applied.
# These four flag layers correspond to FLAG_NAME_LIST indices 0-3.
PRE_AOFLAGGER_FLAGS = [
    "raw_flags_SARAO",
    "raw_flags_noise_diode_on",
    "raw_flags_known_rfi",
    "raw_flags_rawdata_low_value",
]

# Combine all pre-AOFlagger flags into a single boolean array with OR.
flags_before_aoflagger = notebook_helper.combine_flags(ds, PRE_AOFLAGGER_FLAGS)

In [ ]:
def plot_medians_before_aoflagger(
    ds: xr.Dataset, flags_before_aoflagger: xr.DataArray
) -> None:
    raw_vis_before_ao = notebook_helper.select_and_flag(
        ds, "raw_vis", flags=flags_before_aoflagger
    )
    timemedian = raw_vis_before_ao.median(dim="timestamps", skipna=True)
    freqmedian = raw_vis_before_ao.median(dim="frequencies", skipna=True)
    medians = [timemedian, freqmedian]
    del (
        raw_vis_before_ao
    )  # Compute medians on the fly; freed when this function returns.

    freq_arr = timemedian.coords["frequencies"]
    ts = freqmedian.coords["timestamps"]
    # Unix time is in running seconds; convert to minutes for plotting.
    t_min = (ts - ts.min()) / 60.0

    # fig, axes = plt.subplots(2, 2, figsize=(15, 5.5), sharex="col", layout="constrained")
    fig = plt.figure(figsize=(8, 8), constrained_layout=True)
    subfigs = fig.subfigures(2, 1, hspace=0.03)
    axes = [sf.subplots(2, 1, sharex="all") for sf in subfigs]
    for i, (ax, md) in enumerate(zip(axes, medians)):
        x = freq_arr if i == 0 else t_min
        for j, feed in enumerate(["h", "v"]):
            ax[j].plot(x, md.sel(feeds=feed), lw=0.7, rasterized=True)
            add_anchored_text(ax[j], feed, loc="upper right")
        ax[-1].set_xlabel("Frequency [MHz]") if i == 0 else ax[-1].set_xlabel(
            "Time [min]"
        )
        ax[0].set_ylabel("Temperature [uncalibrated]", y=0)
    fig.suptitle(
        f"Time and Frequency Medians of Raw Vis Before AOFlagger - {ds.attrs['block_name']}"
    )
    fig.align_ylabels()


plot_medians_before_aoflagger(ds, flags_before_aoflagger)

## 5. Raw and Calibrated Visbility Time-Median Spectra
Examine the time medians of the flagged raw and calibrated visibilities to assess data quality and evaluate the effectiveness of RFI flagging. Each line is an antenna. All `raw_flags_*` and `cal_flags_*` (see [Data Preparation](#1.-Data-Preparation)) are applied to raw and calibrated visibilities respectively before calculating the medians.

The raw time medians of all antennas should follow the same trend with no obvious spikes and excessive ripples.

The calibrated time medians should largely follow the synchrotron spectrum power law with no spikes or excessive ripples. 

Based on your judgement, note any antennas and/or frequencies that do not follow these trend or have excessive ripples/spikes or large offsets from the rest. 

In [ ]:
def calculate_median(
    ds: xr.Dataset, vis_key: str, dim: str | tuple[str], flag_key: str | None = None
) -> xr.DataArray:
    """Calculate the median of a visibility data variable along specified dimensions.

    Parameters
    ----------
    ds : xr.Dataset
        The input xarray Dataset containing visibility data.
    vis_key : str
        The key of the visibility data variable in the Dataset.
    dim : str or tuple of str
        The dimension(s) along which to calculate the median.
    flag_key : str, optional
        The key of the flag data variable in the Dataset to apply to the visibility
        data before calculating the median. If None, no flagging is applied.

    Returns
    -------
    xr.DataArray
        A new xarray DataArray containing the median values of the specified visibility
        data variable.
    """
    valid_data = notebook_helper.select_and_flag(ds, vis_key, flags=flag_key)
    median_data = valid_data.median(dim=dim, skipna=True)
    return median_data

In [ ]:
def plot_raw_vs_calibrated_spectra(ds: xr.Dataset) -> None:
    """Plot time-averaged median spectra of raw and calibrated visibilities."""
    raw_timemedian = calculate_median(
        ds, vis_key="raw_vis", dim="timestamps", flag_key="raw_flags_combined"
    )  # (frequencies, antennas, feeds) — computed on the fly, freed when this returns.
    cal_timemedian = calculate_median(
        ds, vis_key="cal_vis", dim="timestamps", flag_key="cal_flags_combined"
    ).sel(stokes="I", drop=True)  # (frequencies, antennas)
    freq_arr = raw_timemedian.coords["frequencies"]

    fig, (ax1, ax2, ax3) = plt.subplots(
        3, 1, figsize=(8, 5.5), sharex="all", layout="constrained"
    )
    ax1.plot(freq_arr, raw_timemedian.sel(feeds="h"), lw=0.7, rasterized=True)
    ax2.plot(freq_arr, raw_timemedian.sel(feeds="v"), lw=0.7, rasterized=True)
    ax3.plot(freq_arr, cal_timemedian, lw=0.7, rasterized=True)
    for ax, label in [(ax1, "Raw h"), (ax2, "Raw v"), (ax3, "Calibrated")]:
        add_anchored_text(ax, label, loc="upper right")
    ax1.set_ylabel("Temperature\n[uncalibrated]")
    ax2.set_ylabel("Temperature\n[uncalibrated]")
    ax3.set_ylabel("Temperature\n[$K_{RJ}$]")
    ax3.set_xlabel("Frequency [MHz]")
    fig.suptitle(f"Time Medians of Raw and Calibrated Visibilities - {block_name}")
    fig.align_ylabels()

In [ ]:
plot_raw_vs_calibrated_spectra(ds)

## 6. Mean-Subtracted Time Series of Raw Visibility

Examine the mean-subtracted time series of flagged raw visibility from several randomly selected antennas. All `raw_flags_*` are applied to the `raw_vis` before calculating the mean value (skipping NaNs) that is then subtracted back to the flagged raw visibility (see [Data Preparation](#1.-Data-Preparation) for details of these flags).

This should oscillate as the telescope scan across RA/Dec. Assess consistency across antennas and note any unflagged "rogue" antennas at this stage, whose time series will diverge from the trend.

In [ ]:
def plot_raw_time_series(ds: xr.Dataset, n_freqs: int = 5, n_ants: int = 20) -> None:
    freq_da = ds.coords["frequencies"]
    selected_freqs = np.linspace(
        freq_da.min().values, freq_da.max().values, n_freqs
    )  # Evenly spaced frequencies to plot
    selected_ants = sorted(
        random.sample(ds.coords["antennas"].values.tolist(), n_ants)
    )  # Randomly select antennas to plot
    ts = ds.coords["timestamps"]
    t_min = (ts - ts.min()) / 60.0
    # Apply the combined flags to the raw visibilities, selecting only the chosen
    # antennas and frequencies to keep memory usage minimal.
    vis_flagged = notebook_helper.select_and_flag(
        ds,
        "raw_vis",
        flags="raw_flags_combined",
        sel={"antennas": selected_ants},
        nearest={"frequencies": selected_freqs},
    )
    # Subtract vis_flagged by its mean
    vis_flagged = vis_flagged - vis_flagged.mean(dim="timestamps", skipna=True)

    # Prepare to plot
    fig, axes = plt.subplots(
        len(selected_freqs),
        2,
        figsize=(12, 2 * len(selected_freqs)),
        sharex="all",
        sharey="row",
        layout="constrained",
    )
    # Loop over row (frequency) and plot
    for row, freq in enumerate(selected_freqs):
        for col, feed in enumerate(["h", "v"]):
            axes[row, col].plot(
                t_min,
                vis_flagged.sel(frequencies=freq, feeds=feed),
                lw=0.5,
                alpha=0.7,
                rasterized=True,
            )
            add_anchored_text(
                axes[row, col],
                f"{freq:.1f} MHz - {feed}",
                loc="upper left",
            )
    # Label axes at figure level
    fig.supylabel("Temperature [uncalibrated]")
    fig.supxlabel("Time [min]")
    fig.suptitle(
        f"Mean-subtracted Time Series of Flagged Raw Visibility - {ds.attrs['block_name']}"
    )
    # Add a single legend for all antennas at the top of the figure. Constrained layout
    # does not work with figure legend yet so we have to adjust the padding
    fig.legend(
        handles=axes[0, 0].lines,
        labels=selected_ants,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.98),
        ncol=10,
        frameon=False,
    )
    fig.get_layout_engine().set(rect=[0, 0, 1, 0.95])


plot_raw_time_series(ds)

## 7. Raw Visibility Sky Scatter before AOFlagger

Each row shows the frequency-medians of raw visibilities (with all flags before AOFlagger applied) 
over the scan pointings' (RA, Dec) coordinates for the h and v feeds (left and right columns).
1-Jy and 5-Jy point sources in the field are overlaid in □ and +.
Spearman's correlataion value $\rho$ from non-linearity tests in [Section 3](#3-Non-Linearity-Contamination) for each antenna is also labelled.
The axes and color scale are shared per antenna (per row).
Antennas deemed "bad" in this block have red-colored label.

When reviewing this figure:

- Note unflagged rogue antennas at this stage. All antennas should follow the same scan points, but some antennas may struggle to keep up with the rest, resulting in data points outside the scan area, changing the x-axis range, and/or missing data points in the scan area. 
- Any antennas that exhibit very low or high values.

These antennas should later be flagged in the [Calibrated Visibility vs Synchrotron Model](#8-Median-Subtracted-Calibrated-Visibility-and-Synchrotron-Model-Sky-Scatter) plot.

Also note:

- Any antennas with strong "zebra" pattern and their correlation values.
- Any other unusal artifacts.

In [ ]:
# Load point sources from the catalog for overlaying in sky scatter plots
#
# Define and pass catalog_dir if not running on Ilifu
# catalog_dir = Path.cwd().parent.parent
# point_sources = load_point_sources_for_ds(
#     ds, antenna=good_ants[0], catalog_dir=catalog_dir
# )
#
# It is possible that all antennas are flagged, in which case we can just use "m000"..
point_sources = load_point_sources_for_ds(
    ds, antenna=good_ants[0] if good_ants else "m000"
)

In [ ]:
def plot_raw_sky_median(
    ds: xr.Dataset, flags: xr.DataArray, gsm_corr: xr.DataArray, point_sources: dict
) -> None:
    """
    Plot raw visibility on the sky (RA/Dec) for each antenna and feed, using the median
    over frequencies.

    The axes are shared per row (antenna) to allow mispointings to be visible if exist.
    """
    vis_freqmedian = notebook_helper.select_and_flag(ds, "raw_vis", flags=flags).median(
        dim="frequencies", skipna=True
    )

    # RA/Dec are used unfiltered so unflagged mispointing is visible on the raw maps
    # (and can be checked against the final calibrated maps). Flagged visibility is
    # already NaN, so those points simply don't render -- no separate RA/Dec masking
    # is needed, and lengths stay aligned since ra/dec/vis_freqmedian all share the
    # dataset's "timestamps" coordinate.
    ants = ds.coords["antennas"].values
    all_ants = ds.attrs["all_meerkat_antennas"]

    # Anchored text placement is set up for rising scans (antenna/feed label on top,
    # correlation label on bottom); for setting scans, RA and Dec trends are flipped
    # relative to rising so the vertical placement is swapped to keep labels clear of
    # the scan track.
    vert = "lower" if ds.attrs.get("scan_direction") == "setting" else "upper"
    opp_vert = "upper" if vert == "lower" else "lower"

    # Loop over group of antennas to plot in batches
    for fig_start in range(0, 64, ANTENNAS_PER_FIG):
        # Antennas for this batch
        batch = all_ants[fig_start : fig_start + ANTENNAS_PER_FIG]

        fig, axes = plt.subplots(
            ANTENNAS_PER_FIG,
            2,
            figsize=(10, 1.5 * ANTENNAS_PER_FIG),
            # Each row (antenna) gets its own x/y range instead of one shared across
            # the whole batch: unfiltered RA/Dec can contain mispointing outliers that
            # would otherwise blow out the range for every antenna in the batch.
            sharex="row",
            sharey="row",
            layout="constrained",
        )
        fig.suptitle(
            f"Raw Visibility Frequency Median before AOFlagger - {block_name}",
        )
        for row, ant_name in enumerate(batch):
            if ant_name not in ants:
                for ax in axes[row]:
                    add_anchored_text(
                        ax, f"{ant_name} not in observation", loc="center"
                    )
                    ax.set_axis_off()
                continue

            # RA/Dec for this antenna, unfiltered. Shared by both feed columns since
            # RA/Dec has no feed dependence. RA is unwrapped since it wraps at 360/0
            # and this notebook's fields sit near RA=0.
            ra_ant = notebook_helper.unwrap_ra_deg(
                ds["ra"].sel(antennas=ant_name).values
            )
            dec_ant = ds["dec"].sel(antennas=ant_name).values

            for col, feed in enumerate(["h", "v"]):
                ax = axes[row, col]
                vis_vals = vis_freqmedian.sel(antennas=ant_name, feeds=feed).values
                finite = np.isfinite(vis_vals)
                vmin = float(np.nanmin(vis_vals)) if finite.any() else None
                vmax = float(np.nanmax(vis_vals)) if finite.any() else None
                scatter_sky(
                    ax,
                    ra_ant,
                    dec_ant,
                    vis_vals,
                    vmin=vmin,
                    vmax=vmax,
                    cbar=True if feed == "v" else False,
                    point_sources=point_sources,
                )
                # Label the antenna name and feed on the plot.
                # Good antennas in grey, bad antennas in red.
                add_anchored_text(
                    ax,
                    f"{ant_name}{feed}",
                    loc=f"{vert} left",
                    alpha=1,
                    edgecolor="lightgrey"
                    if ant_name in ds.attrs["good_ants"]
                    else "red",
                    size="small",
                )
                # Label with spearman's correlation from non-linearity test
                spearman_r = gsm_corr.sel(
                    antennas=ant_name, feeds=feed, stats="spearman_r"
                ).values
                add_anchored_text(
                    ax,
                    f"$\\rho_{{RFI}}$={spearman_r:.2f}",
                    loc=f"{opp_vert} right",
                    alpha=1,
                    edgecolor="lightgrey",
                    size="small",
                )

            # Flip RA to match the sky view; invert per row since the x-axis is
            # shared for each row
            axes[row, 0].invert_xaxis()
            # Add 10% margin to the x-axis to accomodate labels
            axes[row, 0].set_xmargin(0.1)

        # Label x/y axes at figure level
        fig.supxlabel("RA [deg]")
        fig.supylabel("Dec [deg]")
        # Adjust the right margin to make space for a shared colorbar label
        fig.get_layout_engine().set(rect=[0, 0, 0.96, 1])
        fig.text(
            0.98,
            0.5,
            "Temperature [uncalibrated]",
            rotation=90,
            va="center",
            ha="right",
            size="large",
        )


plot_raw_sky_median(ds, flags_before_aoflagger, spearman_skysub, point_sources)

## 8. Median-Subtracted Calibrated Visibility and Synchrotron Model Sky Scatter

Compare the median-subtracted calibrated visibility in the left column with a smoothed PySM synchrotron model in the right column over the same sky pointings. Both maps are median-subtracted. The “combined” final calibration flags are applied to the visibility data. Sources brighter than 1 Jy and 5 Jy are overlaid as described above. Spearman's correlation coefficient ($\rho_{synch}$) between the calibrated visibility and the synchrotron model is also labelled per antenna. All antennas deemed bad should be completely flagged and should thus have no correlation values.

In [ ]:
def plot_calibrated_vs_synch(
    ds: xr.Dataset,
    point_sources: dict | None = None,
    freq_plot: float = 730.0,
    freq_half_width: float = 1.5,
    vmin: float = -0.5,
    vmax: float = 1.0,
) -> None:
    ants = ds.coords["antennas"].values
    all_ants = ds.attrs["all_meerkat_antennas"]

    # Anchored text placement is set up for rising scans (antenna label on top,
    # correlation label on bottom); for setting scans, RA and Dec trends are flipped
    # relative to rising so the vertical placement is swapped to keep labels clear of
    # the scan track.
    vert = "lower" if ds.attrs.get("scan_direction") == "setting" else "upper"
    opp_vert = "upper" if vert == "lower" else "lower"

    # -- Prepare cal_vis --
    # Load cal_vis, down select to the plot frequency bandwidth and apply combined flags
    freq_band = slice(freq_plot - freq_half_width, freq_plot + freq_half_width)
    cal_vis_flagged = notebook_helper.select_and_flag(
        ds,
        "cal_vis",
        flags="cal_flags_combined",
        sel={"stokes": "I", "frequencies": freq_band},
    )
    # Calculate median over frequencies and timestamps, skipping NaNs. This will be used for median subtraction.
    cal_vis_flagged_median = cal_vis_flagged.median(
        dim=("timestamps", "frequencies"), skipna=True
    )
    # Down select cal_vis_flagged further for plotting
    cal_vis_flagged = cal_vis_flagged.sel(frequencies=freq_plot, method="nearest")
    # Median subtraction, broadcast rule should apply
    cal_vis_flagged = cal_vis_flagged - cal_vis_flagged_median

    # Synchrotron healpix map at the plot frequecy
    synch_map = get_synchrotron_map(freq_plot)

    # Loop over group of antennas to plot in batches
    for fig_start in range(0, 64, ANTENNAS_PER_FIG):
        batch = all_ants[fig_start : fig_start + ANTENNAS_PER_FIG]

        fig, axes = plt.subplots(
            ANTENNAS_PER_FIG,
            2,
            figsize=(10, 1.5 * ANTENNAS_PER_FIG),
            # Each row (antenna) gets its own x/y range instead of one shared across
            # the whole batch: unfiltered RA/Dec can contain mispointing outliers that
            # would otherwise blow out the range for every antenna in the batch.
            sharex="row",
            sharey="row",
            layout="constrained",
        )

        for row, ant_name in enumerate(batch):
            # If the antenna is not in the observation, add a text to the axes,
            # turn off the axes and skip this loop
            if ant_name not in ants:
                for ax in axes[row]:
                    add_anchored_text(
                        ax, f"{ant_name} not in observation", loc="center"
                    )
                    ax.set_axis_off()
                continue

            # RA/Dec for this antenna, unfiltered for cal_vis plotting. NaNs in properly
            # flagged cal_vis simply won't render, so there is no need to filter RA/Dec.
            # Plus, we want to see unflagged mispointing if it exists. RA is unwrapped
            # once here (rather than inside scatter_sky) so that the full-array and
            # `[valid]`-filtered subset used below land on the same 360-degree branch
            # and stay aligned on the row's shared x-axis.
            ra_ant = notebook_helper.unwrap_ra_deg(
                ds["ra"].sel(antennas=ant_name).values
            )
            dec_ant = ds["dec"].sel(antennas=ant_name).values

            # cal_vis for this antenna -- still masked (NaN) at flagged timestamps.
            cal_vis_plot = cal_vis_flagged.sel(antennas=ant_name).values

            # Get valid timstamps for synchrotron model interpolation plotting.
            # The synchrotron model is defined everywhere on the sky. There can be a
            # case where cal_vis is flagged but RA/Dec are still valid. This won't
            # affect plotting of cal_vis, but we don't want to interpolate the
            # synchrotron model at those timestamps.
            valid = np.isfinite(cal_vis_plot)

            # Get the interpolated synchrotron model values at the RA/Dec
            synch_vals = get_healpix_interp_vals(
                synch_map, ra_ant[valid], dec_ant[valid]
            )
            # Median subtract and convert to K_RJ
            synch_vals = (
                (synch_vals - np.median(synch_vals)) / 1e6
                if len(synch_vals) > 0
                else synch_vals
            )
            # Correction value for labelling
            r_val = float(ds["r_vis_synch_ant"].sel(antennas=ant_name))

            for i, (ax, vals) in enumerate(zip(axes[row], [cal_vis_plot, synch_vals])):
                # Plot maps, shared cbar only on the right column - they should be on the same scale
                scatter_sky(
                    ax,
                    ra_ant if i == 0 else ra_ant[valid],
                    dec_ant if i == 0 else dec_ant[valid],
                    vals,
                    vmin=vmin,
                    vmax=vmax,
                    cbar=True if i == 1 else False,
                    point_sources=point_sources,
                )
                # Antenna label on the left column, correlation coefficient label on the right column
                add_anchored_text(
                    ax,
                    f"{ant_name}" if i == 0 else f"$\\rho_{{synch}}$={r_val:.2f}",
                    loc=f"{vert} left" if i == 0 else f"{opp_vert} right",
                    size="small",
                    alpha=1,
                    edgecolor="lightgrey"
                    if ant_name in ds.attrs["good_ants"]
                    else "red",
                )
            axes[row, 0].invert_xaxis()
            # Add 10% margin to the x-axis to accomodate labels
            axes[row, 0].set_xmargin(0.1)

        # Label x/y axes at figure level
        fig.suptitle(
            "Median-Subtracted Calibrated Visibility and Synchrotron Model - "
            f"{ds.attrs['block_name']}, {freq_plot:.1f} MHz",
        )
        fig.supxlabel("RA [deg]")
        fig.supylabel("Dec [deg]")
        # Adjust the right margin to make space for a shared colorbar label
        fig.get_layout_engine().set(rect=[0, 0, 0.96, 1.0])
        fig.text(
            0.98,
            0.5,
            "Temperature [K]",
            rotation=90,
            va="center",
            ha="right",
            size="large",
        )


plot_calibrated_vs_synch(ds, point_sources)